# LatentSync trên Colab (T4)

Repo: https://github.com/bytedance/LatentSync

## VRAM thật (theo README)
- **LatentSync 1.5: tối thiểu 8 GB** khi suy luận.
- **LatentSync 1.6: tối thiểu 18 GB** khi suy luận (huấn luyện ở 512x512).
- GPU T4 Colab có **16 GB** -> **bản 1.6 không đủ theo README, bản 1.5 chạy được.** Mặc định notebook dùng **1.5**.

## Cách giảm VRAM / thời gian khi cần
- Dùng bản 1.5 (khuyến nghị cho T4).
- Giảm `inference_steps` (README: 20-50; thấp hơn nhanh hơn, hình kém hơn) và để `guidance_scale` 1.0-3.0.
- Giữ cờ `--enable_deepcache` (có trong `inference.sh` chính thức).
- Cắt video thành đoạn ngắn (vài giây đến ~1 phút) rồi ghép lại bằng FFmpeg.
- Nguồn video rõ mặt, một người, mặt thấy rõ; hạ độ phân giải nếu nguồn rất lớn.
- Không có phần "giảm VRAM khi suy luận" riêng trong README; mục giảm VRAM của README chỉ dành cho huấn luyện.

> **Bước 0:** vào `Runtime > Change runtime type > T4 GPU` rồi bấm Save. Phiên Colab miễn phí có giới hạn thời gian; mọi kết quả đều được lưu ra Google Drive để không mất khi phiên bị ngắt.

Cấu trúc thư mục trên Drive (tự tạo ở ô mount):

```
MyDrive/content-factory/
  input/    <- đưa kịch bản .txt, video, audio lên đây
  output/   <- kết quả để tải về Local
  models/   <- trọng số / checkpoint
```

## 1. Cấu hình

In [ ]:
#@title Tham số
VIDEO_FILE = "video.mp4"     #@param {type:"string"}
AUDIO_FILE = "audio.wav"     #@param {type:"string"}
OUT_NAME   = "video_lipsync.mp4"  #@param {type:"string"}
VERSION    = "1.5"           #@param ["1.5", "1.6"]
INFERENCE_STEPS = 20         #@param {type:"integer"}
GUIDANCE_SCALE  = 1.5        #@param {type:"number"}
ENABLE_DEEPCACHE = True      #@param {type:"boolean"}
LATENTSYNC_COMMIT = ""       #@param {type:"string"}

## 2. Mount Drive và kiểm tra GPU

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/content-factory'
IN_DIR, OUT_DIR, MODEL_DIR = f'{BASE}/input', f'{BASE}/output', f'{BASE}/models'
for d in (IN_DIR, OUT_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)
print('Input :', IN_DIR)
print('Output:', OUT_DIR)
print('Models:', MODEL_DIR)

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'KHONG THAY GPU. Hay chon Runtime > Change runtime type > T4 GPU.')

## 3. Cài đặt
`setup_env.sh` của repo dùng conda + Python 3.10.13. Trên Colab ta chạy các lệnh tương đương không có conda: `pip install -r requirements.txt`, `apt install libgl1`, FFmpeg, rồi tải checkpoint từ Hugging Face.

In [ ]:
import subprocess, os
def sh(cmd, cwd=None, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print('LOI:', r.stderr[-2500:])
        if check:
            raise RuntimeError(f'Lenh that bai: {cmd}')
    return r

sh('python --version')  # setup_env.sh dung Python 3.10.13; Colab co the la ban khac. TODO: kiem tra tuong thich requirements.txt
sh('apt-get -y -qq install ffmpeg libgl1')
if not os.path.isdir('/content/LatentSync'):
    sh('git clone https://github.com/bytedance/LatentSync.git /content/LatentSync')
if LATENTSYNC_COMMIT:
    sh(f'git checkout {LATENTSYNC_COMMIT}', cwd='/content/LatentSync')
sh('git rev-parse HEAD', cwd='/content/LatentSync')
sh('pip install -q -r requirements.txt', cwd='/content/LatentSync')

## 4. Tải checkpoint
Theo `setup_env.sh`: `huggingface-cli download ByteDance/LatentSync-1.6 whisper/tiny.pt ...` và `latentsync_unet.pt`. Với bản 1.5 ta đổi repo thành `ByteDance/LatentSync-1.5` (tên repo này **chưa xác minh**).

In [ ]:
import subprocess, os
repo = 'ByteDance/LatentSync-1.5' if VERSION == '1.5' else 'ByteDance/LatentSync-1.6'
# TODO: kiem tra - ten repo HF cua ban 1.5 va ten tep trong do (README chi xac nhan repo 1.6).
ckpt_cache = f'{MODEL_DIR}/latentsync_{VERSION}'
os.makedirs(ckpt_cache, exist_ok=True)
for f in ('whisper/tiny.pt', 'latentsync_unet.pt'):
    cmd = f'huggingface-cli download {repo} {f} --local-dir /content/LatentSync/checkpoints'
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-800:], r.stderr[-800:])
print(subprocess.run('ls -la /content/LatentSync/checkpoints /content/LatentSync/checkpoints/whisper', shell=True, text=True, capture_output=True).stdout)

## 5. Chạy đồng bộ khẩu hình
Lệnh lấy từ `inference.sh` chính thức (có `--enable_deepcache`, 20 bước, guidance 1.5). Cấu hình UNet: bản 1.6 dùng `configs/unet/stage2_512.yaml`; bản 1.5 nhiều khả năng là `configs/unet/stage2.yaml` (**TODO: kiểm tra**).

In [ ]:
import subprocess, os, shutil
video = f'{IN_DIR}/{VIDEO_FILE}'; audio = f'{IN_DIR}/{AUDIO_FILE}'
for p in (video, audio):
    assert os.path.exists(p), f'Khong thay {p}. Hay dua len Drive/input.'
shutil.copy2(video, '/content/in_video.mp4'); shutil.copy2(audio, '/content/in_audio.wav')

unet_cfg = 'configs/unet/stage2_512.yaml' if VERSION == '1.6' else 'configs/unet/stage2.yaml'  # TODO: kiem tra
cmd = ['python', '-m', 'scripts.inference',
       '--unet_config_path', unet_cfg,
       '--inference_ckpt_path', 'checkpoints/latentsync_unet.pt',
       '--inference_steps', str(INFERENCE_STEPS),
       '--guidance_scale', str(GUIDANCE_SCALE),
       '--video_path', '/content/in_video.mp4',
       '--audio_path', '/content/in_audio.wav',
       '--video_out_path', '/content/out.mp4']
if ENABLE_DEEPCACHE:
    cmd.insert(-8, '--enable_deepcache')
print(' '.join(cmd))
r = subprocess.run(cmd, cwd='/content/LatentSync', text=True, capture_output=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('LOI:', r.stderr[-4000:])
    if 'out of memory' in r.stderr.lower():
        print('>> Het VRAM: dung ban 1.5, giam INFERENCE_STEPS, hoac cat video ngan hon.')
else:
    dst = f'{OUT_DIR}/{OUT_NAME}'
    shutil.copy2('/content/out.mp4', dst)
    print('KET QUA:', dst, f'({os.path.getsize(dst)/1e6:.1f} MB)')